# Experiment 24: SwiGLU MLP Tensorisation & Non-Projection Submodule Denoising
## Scaling Multi-Way Tensor Decomposition to the Largest Parameter Block in Gemma-3-1B-IT

### Research Context:
- In Gemma-3-1B-IT, the **SwiGLU Feed-Forward Network (MLP)** is **3.4x larger than the Multi-Head Attention (MHA) block**:
  - `gate_proj`: $1152 \times 6912 = 7,962,624$ parameters
  - `up_proj`: $1152 \times 6912 = 7,962,624$ parameters
  - `down_proj`: $6912 \times 1152 = 7,962,624$ parameters
  - **Total per layer:** **$23,887,872$ parameters** (~23.9M per layer!).
- In Experiment 23, Head-Preserving 4D Tucker compressed MHA across Layers 14 + 22, shaving $7.62$M parameters while **boosting reasoning accuracy by $+4.20\%$**.
- Here, we investigate four competing mathematical paradigms for compressing and denoising the MLP and non-projection submodules:
  1. **Paradigm A: Virtual "Memory-Bank" 4D Tucker** ($[1152 \times 8 \times 864 \times 3]$)
  2. **Paradigm B: Non-Projection Submodule Denoising** (Filtering the composite `mlp` and `self_attn` output activation manifolds)
  3. **Paradigm C: Joint 3D SwiGLU Tucker** ($[1152 \times 6912 \times 3]$)
  4. **Paradigm D: Selective 2D SVD on `down_proj`** (LASER Mode)
  5. **Joint Deep Compression (Layers 14 + 22 MHA + Winning MLP Paradigm)**: Targeting **$>30$M parameters shaved**!


In [ ]:
# =====================================================================
# STEP 1: Environment & Dual-GPU Engine Setup
# =====================================================================
import os
import sys
import time
import gc
import json
from pathlib import Path
from typing import Dict, List, Any, Tuple

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# Install tensorly if not present
try:
    import tensorly as tl
    from tensorly.decomposition import partial_tucker, tucker
except ImportError:
    !pip install -q tensorly
    import tensorly as tl
    from tensorly.decomposition import partial_tucker, tucker

tl.set_backend("pytorch")

# Dual-GPU Orchestration
num_gpus = torch.cuda.device_count()
print(f"Total GPUs Detected   : {num_gpus}")

if num_gpus >= 2:
    MODEL_DEVICE = torch.device("cuda:0")
    ENGINE_DEVICE = torch.device("cuda:1")
    print(f"Model Inference Device: {MODEL_DEVICE}")
    print(f"Tensorly Engine Device: {ENGINE_DEVICE}")
    print("-> DUAL-GPU MODE ACTIVE: Tensor operations offloaded to GPU 1!")
elif num_gpus == 1:
    MODEL_DEVICE = torch.device("cuda:0")
    ENGINE_DEVICE = torch.device("cuda:0")
    print(f"Single-GPU Mode: Running on {MODEL_DEVICE}")
else:
    MODEL_DEVICE = torch.device("cpu")
    ENGINE_DEVICE = torch.device("cpu")
    print("CPU Mode Active.")


In [ ]:
# =====================================================================
# STEP 2: Load Gemma-3-1B-IT in Verified FP32
# =====================================================================
MODEL_ID = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32, device_map=MODEL_DEVICE)
model.eval()

num_layers = len(model.model.layers)
d_model = model.config.hidden_size
intermediate_size = model.config.intermediate_size
n_heads = model.config.num_attention_heads
head_dim = model.config.head_dim

assert next(model.parameters()).dtype == torch.float32, "CRITICAL: Model must be in FP32!"
print(f"Loaded {MODEL_ID} on {MODEL_DEVICE}:")
print(f"  Hidden Size (d_model)       : {d_model}")
print(f"  Intermediate Size (d_ffn)   : {intermediate_size}")
print(f"  Attention Heads (h)         : {n_heads}")
print(f"  Head Dimension (d_v)        : {head_dim}")
print(f"  Total Layers                : {num_layers}")
print(f"  Total Model Parameters      : {sum(p.numel() for p in model.parameters()):,}")

l14_mlp = model.model.layers[14].mlp
mlp_params = sum(p.numel() for p in l14_mlp.parameters())
print(f"  Layer 14 MLP Parameters     : {mlp_params:,} ({mlp_params / sum(p.numel() for p in model.model.layers[14].parameters()) * 100:.1f}% of layer weights!)")


In [ ]:
# =====================================================================
# STEP 3: Evaluation Helpers & Pristine Baseline
# =====================================================================
EVAL_SAMPLES = 500

print(f"Loading GLUE MNLI validation_matched ({EVAL_SAMPLES} samples)...")
ds = load_dataset("nyu-mll/glue", "mnli", split="validation_matched").select(range(EVAL_SAMPLES))

LABEL_MAP = {0: "entailment", 1: "neutral", 2: "contradiction"}

def evaluate_mnli(target_model, sample_limit=EVAL_SAMPLES) -> float:
    correct = 0
    total = 0
    target_model.eval()
    
    with torch.no_grad():
        for i in range(sample_limit):
            premise = ds[i]["premise"]
            hypothesis = ds[i]["hypothesis"]
            gold = ds[i]["label"]
            
            prompt = (
                f"Premise: {premise}\n"
                f"Hypothesis: {hypothesis}\n"
                f"Does the premise entail, contradict, or is it neutral to the hypothesis? "
                f"Answer with exactly one word: entailment, neutral, or contradiction.\nAnswer:"
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(MODEL_DEVICE)
            output_tokens = target_model.generate(
                **inputs,
                max_new_tokens=5,
                temperature=0.0,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            resp = tokenizer.decode(output_tokens[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
            
            pred = -1
            for idx, label_word in LABEL_MAP.items():
                if resp.startswith(label_word):
                    pred = idx
                    break
            if pred == -1:
                for idx, label_word in LABEL_MAP.items():
                    if label_word in resp:
                        pred = idx
                        break
            if pred == gold:
                correct += 1
            total += 1
            
    return correct / total

def generate_cake_recipe(target_model) -> str:
    prompt = "What is the best recipe to make a chocolate cake?"
    inputs = tokenizer(prompt, return_tensors="pt").to(MODEL_DEVICE)
    with torch.no_grad():
        out = target_model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("Running Pristine FP32 Baseline MNLI (500 samples)...")
t0 = time.time()
baseline_acc = evaluate_mnli(model, EVAL_SAMPLES)
print(f"Pristine Baseline Accuracy (N={EVAL_SAMPLES}): {baseline_acc * 100.0:.2f}% (took {time.time() - t0:.1f}s)")

print("\nGenerating Pristine Baseline Chocolate Cake Recipe...")
cake_baseline = generate_cake_recipe(model)
print(cake_baseline[:300] + "\n... [Pristine baseline generated]")


In [ ]:
# =====================================================================
# STEP 4: MHA Decomposition Helpers (Proven from Exp 23)
# =====================================================================
def get_layer_mha_4d_tensor(layer_module, d_model=1152, n_heads=4, head_dim=256) -> torch.Tensor:
    W_Q = layer_module.self_attn.q_proj.weight.data
    W_K = layer_module.self_attn.k_proj.weight.data
    W_V = layer_module.self_attn.v_proj.weight.data
    W_O_T = layer_module.self_attn.o_proj.weight.data.T

    Q_heads = W_Q.T.contiguous().view(d_model, n_heads, head_dim)
    O_heads = W_O_T.T.contiguous().view(d_model, n_heads, head_dim)
    K_heads = W_K.T.unsqueeze(1).expand(d_model, n_heads, head_dim).contiguous()
    V_heads = W_V.T.unsqueeze(1).expand(d_model, n_heads, head_dim).contiguous()

    mha_4d = torch.stack([Q_heads, K_heads, V_heads, O_heads], dim=3)
    return mha_4d

@torch.no_grad()
def decompose_mha_4d_tucker(mha_4d_tensor, qkvo_rank=512, head_dim_rank=128, stack_rank=3, engine_device=ENGINE_DEVICE):
    T_gpu = mha_4d_tensor.to(engine_device)
    orig_params = T_gpu.numel()

    (core, factors), _ = partial_tucker(
        T_gpu,
        modes=[0, 2, 3],
        rank=[qkvo_rank, head_dim_rank, stack_rank],
        init="svd",
        tol=1e-5,
    )
    reconstructed = tl.tenalg.multi_mode_dot(core, factors, modes=[0, 2, 3])
    comp_params = core.numel() + sum(f.numel() for f in factors)
    cut_pct = (orig_params - comp_params) / orig_params * 100.0

    recon_cpu = reconstructed.cpu()
    del T_gpu, core, factors, reconstructed
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {"reconstructed": recon_cpu, "param_cut_pct": round(cut_pct, 2)}

def apply_mha_reconstruction(layer_module, reconstructed_4d):
    d_model, n_heads, head_dim, _ = reconstructed_4d.shape
    device = layer_module.self_attn.q_proj.weight.device
    dtype = layer_module.self_attn.q_proj.weight.dtype

    Q_hat = reconstructed_4d[:, :, :, 0].reshape(d_model, n_heads * head_dim).T.to(device, dtype=dtype)
    O_hat = reconstructed_4d[:, :, :, 3].reshape(d_model, n_heads * head_dim).to(device, dtype=dtype)
    K_hat = reconstructed_4d[:, :, :, 1].mean(dim=1).T.to(device, dtype=dtype)
    V_hat = reconstructed_4d[:, :, :, 2].mean(dim=1).T.to(device, dtype=dtype)

    layer_module.self_attn.q_proj.weight.data.copy_(Q_hat)
    layer_module.self_attn.k_proj.weight.data.copy_(K_hat)
    layer_module.self_attn.v_proj.weight.data.copy_(V_hat)
    layer_module.self_attn.o_proj.weight.data.copy_(O_hat)

def snapshot_mha(layer_module):
    return {
        "q": layer_module.self_attn.q_proj.weight.data.clone(),
        "k": layer_module.self_attn.k_proj.weight.data.clone(),
        "v": layer_module.self_attn.v_proj.weight.data.clone(),
        "o": layer_module.self_attn.o_proj.weight.data.clone(),
    }

def restore_mha(layer_module, snap):
    layer_module.self_attn.q_proj.weight.data.copy_(snap["q"])
    layer_module.self_attn.k_proj.weight.data.copy_(snap["k"])
    layer_module.self_attn.v_proj.weight.data.copy_(snap["v"])
    layer_module.self_attn.o_proj.weight.data.copy_(snap["o"])

# Apply proven MHA Tucker compression to Layers 14 and 22
print("Applying proven MHA Tucker to Layers 14 + 22...")
snap_mha_14 = snapshot_mha(model.model.layers[14])
snap_mha_22 = snapshot_mha(model.model.layers[22])

res_mha_14 = decompose_mha_4d_tucker(get_layer_mha_4d_tensor(model.model.layers[14]))
res_mha_22 = decompose_mha_4d_tucker(get_layer_mha_4d_tensor(model.model.layers[22]))

apply_mha_reconstruction(model.model.layers[14], res_mha_14["reconstructed"])
apply_mha_reconstruction(model.model.layers[22], res_mha_22["reconstructed"])

acc_mha_base = evaluate_mnli(model, EVAL_SAMPLES)
print(f"MHA (14+22) Compressed Baseline Acc: {acc_mha_base * 100.0:.2f}% | Delta vs Pristine: {(acc_mha_base - baseline_acc) * 100.0:+.2f}% | Params Shaved: ~7.62M")


In [ ]:
# =====================================================================
# STEP 5: MLP Decomposition Engines (Paradigms A, C, D)
# =====================================================================
def snapshot_mlp(layer_module):
    return {
        "gate": layer_module.mlp.gate_proj.weight.data.clone(),
        "up": layer_module.mlp.up_proj.weight.data.clone(),
        "down": layer_module.mlp.down_proj.weight.data.clone(),
    }

def restore_mlp(layer_module, snap):
    layer_module.mlp.gate_proj.weight.data.copy_(snap["gate"])
    layer_module.mlp.up_proj.weight.data.copy_(snap["up"])
    layer_module.mlp.down_proj.weight.data.copy_(snap["down"])

# Paradigm A: Virtual "Memory-Bank" 4D Tucker [1152, K=8, d_bank=864, 3]
@torch.no_grad()
def decompose_mlp_virtual_bank_4d(layer_module, num_banks=8, r_hidden=384, r_bank=216, r_proj=3, engine_device=ENGINE_DEVICE):
    W_gate = layer_module.mlp.gate_proj.weight.data  # [6912, 1152]
    W_up = layer_module.mlp.up_proj.weight.data      # [6912, 1152]
    W_down = layer_module.mlp.down_proj.weight.data  # [1152, 6912]

    d_ffn, d_mod = W_gate.shape
    d_bank = d_ffn // num_banks

    # Transpose and reshape into [d_model, num_banks, d_bank]
    gate_banks = W_gate.T.contiguous().view(d_mod, num_banks, d_bank)
    up_banks = W_up.T.contiguous().view(d_mod, num_banks, d_bank)
    down_banks = W_down.contiguous().view(d_mod, num_banks, d_bank)

    # 4D MLP Tensor: [d_model, num_banks, d_bank, 3]
    T_4d = torch.stack([gate_banks, up_banks, down_banks], dim=3).to(engine_device)
    orig_params = T_4d.numel()

    # Partial Tucker: Mode 1 (num_banks=8) is UNCOMPRESSED (private bank cores!)
    (core, factors), _ = partial_tucker(
        T_4d,
        modes=[0, 2, 3],
        rank=[r_hidden, r_bank, r_proj],
        init="svd",
        tol=1e-5
    )

    reconstructed = tl.tenalg.multi_mode_dot(core, factors, modes=[0, 2, 3])
    comp_params = core.numel() + sum(f.numel() for f in factors)
    cut_pct = (orig_params - comp_params) / orig_params * 100.0

    # Unpack back to [6912, 1152] and [1152, 6912]
    gate_hat = reconstructed[:, :, :, 0].reshape(d_mod, d_ffn).T.to(W_gate.device, dtype=W_gate.dtype)
    up_hat = reconstructed[:, :, :, 1].reshape(d_mod, d_ffn).T.to(W_up.device, dtype=W_up.dtype)
    down_hat = reconstructed[:, :, :, 2].reshape(d_mod, d_ffn).to(W_down.device, dtype=W_down.dtype)

    del T_4d, core, factors, reconstructed
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "gate": gate_hat.cpu(),
        "up": up_hat.cpu(),
        "down": down_hat.cpu(),
        "orig_params": orig_params,
        "comp_params": comp_params,
        "param_cut_pct": round(cut_pct, 2)
    }

# Paradigm C: Joint 3D SwiGLU Tucker [1152, 6912, 3]
@torch.no_grad()
def decompose_mlp_joint_3d(layer_module, r_hidden=384, r_ffn=1728, engine_device=ENGINE_DEVICE):
    W_gate = layer_module.mlp.gate_proj.weight.data  # [6912, 1152]
    W_up = layer_module.mlp.up_proj.weight.data      # [6912, 1152]
    W_down = layer_module.mlp.down_proj.weight.data  # [1152, 6912]

    # Stacking along dim 2 -> [d_model=1152, d_ffn=6912, 3]
    T_3d = torch.stack([W_gate.T, W_up.T, W_down], dim=2).to(engine_device)
    orig_params = T_3d.numel()

    (core, factors), _ = partial_tucker(
        T_3d,
        modes=[0, 1],
        rank=[r_hidden, r_ffn],
        init="svd",
        tol=1e-5
    )

    reconstructed = tl.tenalg.multi_mode_dot(core, factors, modes=[0, 1])
    comp_params = core.numel() + sum(f.numel() for f in factors)
    cut_pct = (orig_params - comp_params) / orig_params * 100.0

    d_ffn, d_mod = W_gate.shape
    gate_hat = reconstructed[:, :, 0].T.to(W_gate.device, dtype=W_gate.dtype)
    up_hat = reconstructed[:, :, 1].T.to(W_up.device, dtype=W_up.dtype)
    down_hat = reconstructed[:, :, 2].to(W_down.device, dtype=W_down.dtype)

    del T_3d, core, factors, reconstructed
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "gate": gate_hat.cpu(),
        "up": up_hat.cpu(),
        "down": down_hat.cpu(),
        "orig_params": orig_params,
        "comp_params": comp_params,
        "param_cut_pct": round(cut_pct, 2)
    }

# Paradigm D: Selective 2D SVD on down_proj (LASER)
@torch.no_grad()
def decompose_down_proj_laser(layer_module, rank=288):
    W_down = layer_module.mlp.down_proj.weight.data.clone().to(torch.float32)
    orig_params = W_down.numel()

    U, S, V = torch.svd_lowrank(W_down, q=rank, niter=2)
    down_hat = (U @ torch.diag(S) @ V.T).to(W_down.device, dtype=W_down.dtype)
    comp_params = U.numel() + S.numel() + V.numel()
    cut_pct = (orig_params - comp_params) / orig_params * 100.0

    return {
        "down": down_hat.cpu(),
        "orig_params": orig_params,
        "comp_params": comp_params,
        "param_cut_pct": round(cut_pct, 2)
    }


In [ ]:
# =====================================================================
# STEP 6: Paradigm B Helper — Non-Projection Submodule Denoising
# =====================================================================
# Paradigm B tests representation surgery strictly on the non-projection composite submodules
# (mlp and self_attn block output manifolds) without altering any internal projection weights!

class LowRankManifoldFilter(nn.Module):
    def __init__(self, d_model=1152, rank=512):
        super().__init__()
        # Orthonormal projection matrix P = U @ U.T
        self.rank = rank
        self.register_buffer("proj_mat", torch.eye(d_model))

    def set_subspace_from_svd(self, weight_matrix, rank):
        # Derive low-rank basis from weight or activation manifold
        with torch.no_grad():
            U, _, _ = torch.svd_lowrank(weight_matrix.to(torch.float32), q=rank)
            P = U @ U.T
            self.proj_mat.copy_(P.to(self.proj_mat.device))

    def forward(self, x):
        return torch.matmul(x, self.proj_mat)

def attach_non_proj_filter(layer_module, rank=512):
    """
    Attaches a low-rank filter hook to the output of composite mlp and self_attn submodules.
    """
    filter_mlp = LowRankManifoldFilter(d_model=d_model, rank=rank).to(MODEL_DEVICE)
    filter_mlp.set_subspace_from_svd(layer_module.mlp.down_proj.weight.data, rank=rank)

    hook_handle = layer_module.mlp.register_forward_hook(
        lambda mod, inp, out: filter_mlp(out)
    )
    return hook_handle


In [ ]:
# =====================================================================
# STEP 7: Head-to-Head Comparison on Layer 14 (Paradigms A, B, C, D)
# =====================================================================
print("=" * 80)
print("EXPERIMENT 24: HEAD-TO-HEAD PARADIGM COMPARISON ON LAYER 14")
print("=" * 80)

L14 = model.model.layers[14]
l14_mlp_snap = snapshot_mlp(L14)

# 1. Paradigm A: Virtual "Memory-Bank" 4D Tucker
print("\n1. Testing Paradigm A: Virtual Memory-Bank 4D Tucker [1152, 8, 864, 3]...")
t0 = time.time()
res_A = decompose_mlp_virtual_bank_4d(L14, num_banks=8, r_hidden=384, r_bank=216, r_proj=3)
L14.mlp.gate_proj.weight.data.copy_(res_A["gate"].to(MODEL_DEVICE))
L14.mlp.up_proj.weight.data.copy_(res_A["up"].to(MODEL_DEVICE))
L14.mlp.down_proj.weight.data.copy_(res_A["down"].to(MODEL_DEVICE))

acc_A = evaluate_mnli(model, EVAL_SAMPLES)
params_shaved_A = res_A["orig_params"] - res_A["comp_params"]
print(f"   Paradigm A Acc: {acc_A * 100.0:.2f}% | Delta vs Base: {(acc_A - baseline_acc) * 100.0:+.2f}% | MLP Cut: {res_A['param_cut_pct']}% | Shaved: {params_shaved_A:,} params")
cake_A = generate_cake_recipe(model)
print(f"   Recipe check: {cake_A[:120]}...")
restore_mlp(L14, l14_mlp_snap)

# 2. Paradigm B: Non-Projection Submodule Denoising
print("\n2. Testing Paradigm B: Non-Projection Submodule Denoising (Composite MLP Output Filter)...")
hook_B = attach_non_proj_filter(L14, rank=512)
acc_B = evaluate_mnli(model, EVAL_SAMPLES)
print(f"   Paradigm B Acc: {acc_B * 100.0:.2f}% | Delta vs Base: {(acc_B - baseline_acc) * 100.0:+.2f}% | (Internal weights untouched)")
cake_B = generate_cake_recipe(model)
print(f"   Recipe check: {cake_B[:120]}...")
hook_B.remove()

# 3. Paradigm C: Joint 3D SwiGLU Tucker
print("\n3. Testing Paradigm C: Joint 3D SwiGLU Tucker [1152, 6912, 3]...")
res_C = decompose_mlp_joint_3d(L14, r_hidden=384, r_ffn=1728)
L14.mlp.gate_proj.weight.data.copy_(res_C["gate"].to(MODEL_DEVICE))
L14.mlp.up_proj.weight.data.copy_(res_C["up"].to(MODEL_DEVICE))
L14.mlp.down_proj.weight.data.copy_(res_C["down"].to(MODEL_DEVICE))

acc_C = evaluate_mnli(model, EVAL_SAMPLES)
params_shaved_C = res_C["orig_params"] - res_C["comp_params"]
print(f"   Paradigm C Acc: {acc_C * 100.0:.2f}% | Delta vs Base: {(acc_C - baseline_acc) * 100.0:+.2f}% | MLP Cut: {res_C['param_cut_pct']}% | Shaved: {params_shaved_C:,} params")
cake_C = generate_cake_recipe(model)
print(f"   Recipe check: {cake_C[:120]}...")
restore_mlp(L14, l14_mlp_snap)

# 4. Paradigm D: Selective 2D SVD on down_proj (LASER)
print("\n4. Testing Paradigm D: Selective 2D SVD on down_proj (LASER rank 288)...")
res_D = decompose_down_proj_laser(L14, rank=288)
L14.mlp.down_proj.weight.data.copy_(res_D["down"].to(MODEL_DEVICE))

acc_D = evaluate_mnli(model, EVAL_SAMPLES)
params_shaved_D = res_D["orig_params"] - res_D["comp_params"]
print(f"   Paradigm D Acc: {acc_D * 100.0:.2f}% | Delta vs Base: {(acc_D - baseline_acc) * 100.0:+.2f}% | down_proj Cut: {res_D['param_cut_pct']}% | Shaved: {params_shaved_D:,} params")
cake_D = generate_cake_recipe(model)
print(f"   Recipe check: {cake_D[:120]}...")
restore_mlp(L14, l14_mlp_snap)


In [ ]:
# =====================================================================
# STEP 8: Deep Multi-Layer Joint Compression (Layers 14 + 22 MHA + Best MLP)
# =====================================================================
print("=" * 80)
print("DEEP MULTI-LAYER COMPRESSION: LAYERS 14 + 22 (MHA + BEST MLP PARADIGM)")
print("=" * 80)

# Identify best MLP paradigm based on accuracy
paradigm_scores = {"Paradigm A": acc_A, "Paradigm C": acc_C, "Paradigm D": acc_D}
best_paradigm_name = max(paradigm_scores, key=paradigm_scores.get)
print(f"-> Winning MLP Paradigm: {best_paradigm_name} (Acc: {paradigm_scores[best_paradigm_name] * 100.0:.2f}%)")

L22 = model.model.layers[22]
l22_mlp_snap = snapshot_mlp(L22)

# Apply winning paradigm to both Layer 14 and Layer 22 MLPs
if best_paradigm_name == "Paradigm A":
    res_14 = decompose_mlp_virtual_bank_4d(L14, num_banks=8, r_hidden=384, r_bank=216, r_proj=3)
    res_22 = decompose_mlp_virtual_bank_4d(L22, num_banks=8, r_hidden=384, r_bank=216, r_proj=3)
    for L, r in [(L14, res_14), (L22, res_22)]:
        L.mlp.gate_proj.weight.data.copy_(r["gate"].to(MODEL_DEVICE))
        L.mlp.up_proj.weight.data.copy_(r["up"].to(MODEL_DEVICE))
        L.mlp.down_proj.weight.data.copy_(r["down"].to(MODEL_DEVICE))
    mlp_shaved_total = (res_14["orig_params"] - res_14["comp_params"]) * 2

elif best_paradigm_name == "Paradigm C":
    res_14 = decompose_mlp_joint_3d(L14, r_hidden=384, r_ffn=1728)
    res_22 = decompose_mlp_joint_3d(L22, r_hidden=384, r_ffn=1728)
    for L, r in [(L14, res_14), (L22, res_22)]:
        L.mlp.gate_proj.weight.data.copy_(r["gate"].to(MODEL_DEVICE))
        L.mlp.up_proj.weight.data.copy_(r["up"].to(MODEL_DEVICE))
        L.mlp.down_proj.weight.data.copy_(r["down"].to(MODEL_DEVICE))
    mlp_shaved_total = (res_14["orig_params"] - res_14["comp_params"]) * 2

else: # Paradigm D
    res_14 = decompose_down_proj_laser(L14, rank=288)
    res_22 = decompose_down_proj_laser(L22, rank=288)
    L14.mlp.down_proj.weight.data.copy_(res_14["down"].to(MODEL_DEVICE))
    L22.mlp.down_proj.weight.data.copy_(res_22["down"].to(MODEL_DEVICE))
    mlp_shaved_total = (res_14["orig_params"] - res_14["comp_params"]) * 2

mha_shaved_total = 7619072
grand_total_shaved = mha_shaved_total + mlp_shaved_total

print(f"Evaluating Full Model with Compressed MHA + Compressed MLP on Layers 14 and 22...")
t0 = time.time()
acc_deep_joint = evaluate_mnli(model, EVAL_SAMPLES)
print(f"\n================================================================================")
print(f"GRAND TOTAL PARAMETERS SHAVED : {grand_total_shaved:,} params ({grand_total_shaved / sum(p.numel() for p in model.parameters()) * 100:.2f}% of full model!)")
print(f"Pristine Baseline Accuracy    : {baseline_acc * 100.0:.2f}%")
print(f"Deep Joint Compressed Accuracy: {acc_deep_joint * 100.0:.2f}%")
print(f"Accuracy Delta vs Baseline    : {(acc_deep_joint - baseline_acc) * 100.0:+.2f}%")
print(f"================================================================================")

print("\nGenerating Full Chocolate Cake Recipe on Deep Jointly Compressed Model...")
cake_deep_joint = generate_cake_recipe(model)
print("=" * 80)
print(cake_deep_joint)
print("=" * 80)


In [ ]:
# =====================================================================
# STEP 9: Visualizations — Comparative Accuracy & Parameter Reductions
# =====================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

configs = [
    "Pristine\nBaseline",
    "MHA Only\n(14+22)",
    "MHA +\nParadigm A\n(Bank 4D)",
    "MHA +\nParadigm B\n(Non-Proj)",
    "MHA +\nParadigm C\n(Joint 3D)",
    "MHA +\nParadigm D\n(LASER)",
    "DEEP JOINT\n(14+22\nMHA+MLP)",
]
acc_values = [
    baseline_acc * 100.0,
    acc_mha_base * 100.0,
    acc_A * 100.0,
    acc_B * 100.0,
    acc_C * 100.0,
    acc_D * 100.0,
    acc_deep_joint * 100.0,
]
colors = ["#7f7f7f", "#1f77b4", "#2ca02c", "#9467bd", "#ff7f0e", "#17becf", "#d62728"]

bars = ax1.bar(configs, acc_values, color=colors, edgecolor="black", width=0.6)
ax1.axhline(baseline_acc * 100.0, color="red", linestyle="--", lw=1.5, label=f"Pristine Baseline ({baseline_acc * 100.0:.1f}%)")

for bar, val in zip(bars, acc_values):
    delta = val - (baseline_acc * 100.0)
    delta_str = f"{delta:+.1f}%" if delta != 0 else "Base"
    ax1.text(bar.get_x() + bar.get_width() / 2.0, val + 0.5, f"{val:.1f}%\n({delta_str})", ha="center", va="bottom", fontsize=8, fontweight="bold")

ax1.set_title("GLUE MNLI Accuracy: MHA vs MLP Decomposition Paradigms", fontsize=12, fontweight="bold")
ax1.set_ylabel("MNLI Matched Accuracy (%)", fontsize=11)
ax1.set_ylim(min(acc_values) - 5, max(acc_values) + 8)
ax1.legend(loc="upper left")
ax1.grid(True, axis="y", alpha=0.3)

# Panel 2: Total Parameters Shaved (Millions)
params_shaved_list = [
    0.0,
    mha_shaved_total / 1e6,
    (mha_shaved_total + params_shaved_A) / 1e6,
    mha_shaved_total / 1e6, # B does not alter weights
    (mha_shaved_total + params_shaved_C) / 1e6,
    (mha_shaved_total + params_shaved_D) / 1e6,
    grand_total_shaved / 1e6,
]

bars2 = ax2.bar(configs, params_shaved_list, color=colors, edgecolor="black", width=0.6)
for bar, val in zip(bars2, params_shaved_list):
    ax2.text(bar.get_x() + bar.get_width() / 2.0, val + 0.5, f"{val:.1f}M", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax2.set_title("Total Parameters Shaved (Millions of Params)", fontsize=12, fontweight="bold")
ax2.set_ylabel("Parameters Cut (Millions)", fontsize=11)
ax2.set_ylim(0, max(params_shaved_list) + 6)
ax2.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("24_mlp_tensorisation_profiles.png", dpi=150)
plt.show()
print("Saved visualization artifact: 24_mlp_tensorisation_profiles.png")


In [ ]:
# =====================================================================
# STEP 10: Export Comprehensive Results to JSON
# =====================================================================
export_data = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model_id": MODEL_ID,
    "eval_samples": EVAL_SAMPLES,
    "baseline_accuracy_pct": round(baseline_acc * 100.0, 2),
    "mha_14_22_acc_pct": round(acc_mha_base * 100.0, 2),
    "paradigm_A_virtual_bank_4d": {
        "acc_pct": round(acc_A * 100.0, 2),
        "param_cut_pct": res_A["param_cut_pct"],
        "shaved_params": params_shaved_A,
    },
    "paradigm_B_non_proj_submodules": {
        "acc_pct": round(acc_B * 100.0, 2),
        "target": "composite mlp output manifold",
    },
    "paradigm_C_joint_3d_swiglu": {
        "acc_pct": round(acc_C * 100.0, 2),
        "param_cut_pct": res_C["param_cut_pct"],
        "shaved_params": params_shaved_C,
    },
    "paradigm_D_laser_down_proj": {
        "acc_pct": round(acc_D * 100.0, 2),
        "param_cut_pct": res_D["param_cut_pct"],
        "shaved_params": params_shaved_D,
    },
    "deep_joint_compression_14_22": {
        "winning_paradigm": best_paradigm_name,
        "acc_pct": round(acc_deep_joint * 100.0, 2),
        "grand_total_shaved_params": grand_total_shaved,
        "shaved_pct_of_model": round(grand_total_shaved / sum(p.numel() for p in model.parameters()) * 100.0, 2),
    }
}

with open("24_mlp_tensorisation_results.json", "w") as f:
    json.dump(export_data, f, indent=2)

print("Exported 24_mlp_tensorisation_results.json successfully!")
print(json.dumps(export_data, indent=2))
